In [1]:
import numpy as np


In [2]:

# Constants used in the equations
KB = 1.38066e-23       # Boltzmann constant [J K-1]
G = 9.81               # gravitational acceleration [m s-2]

DUST_DIAMETERS_UM = np.array([1.46, 2.80, 4.80, 9.00, 16.0])




In [3]:
# terminal velocity of cloud droplets and raindrops

def hydrometeor_terminal_velocity(diameter_um, hydrometeor):
    """Calculate the terminal velocity of a cloud droplet or raindrop."""

    if diameter_um <= 0.0:
        raise ValueError("diameter_um must be greater than zero")

    diameter_m = diameter_um * 1.0e-6

    if hydrometeor == "cloud":
        a = 3.0e7
        b = 2.0
    elif hydrometeor == "rain":
        a = 841.997
        b = 0.8
    else:
        raise ValueError("hydrometeor must be 'cloud' or 'rain'")

    return a * diameter_m**b


# calculating Cunningham correction factor

def cunningham_correction(dp_um):
    """Calculate Cc. The particle diameter in this expression is in micrometres."""

    if dp_um <= 0.0:
        raise ValueError("dp_um must be greater than zero")

    return 1.0 + 0.167 / dp_um


# Particle diffusivity

def particle_diffusivity(dp_um, temperature, mu_air):
    """Calculate particle diffusivity D [m2 s-1]."""

    if temperature <= 0.0 or mu_air <= 0.0:
        raise ValueError("temperature and mu_air must be greater than zero")

    dp_m = dp_um * 1.0e-6
    Cc = cunningham_correction(dp_um)

    D = KB * temperature * Cc / (3.0 * np.pi * mu_air * dp_m)
    return D


# Dust particle terminal velocity

def particle_terminal_velocity(dp_um, mu_air):
    """Calculate particle terminal velocity """

    if dp_um <= 0.0 or mu_air <= 0.0:
        raise ValueError("dp_um and mu_air must be greater than zero")

    dp_m = dp_um * 1.0e-6
    Cc = cunningham_correction(dp_um)

    vt = Cc * G / (3.0 * np.pi * mu_air * dp_m)
    return vt


# Particle relaxation time

def particle_relaxation_time(dp_um, mu_air):
    """Calculate particle relaxation time tau [s]."""

    vt = particle_terminal_velocity(dp_um, mu_air)
    return vt / G


# Reynolds number of a raindrop

def reynolds_number(dd_um, rho_air, mu_air):
    """Calculate Re = dd * Vt(dd) * rho_air / (2 * mu_air)."""

    if dd_um <= 0.0 or rho_air <= 0.0 or mu_air <= 0.0:
        raise ValueError("dd_um, rho_air and mu_air must be greater than zero")

    dd_m = dd_um * 1.0e-6
    Vt_rain = hydrometeor_terminal_velocity(dd_um, "rain")

    Re = dd_m * Vt_rain * rho_air / (2.0 * mu_air)
    return Re


# Schmidt number

def schmidt_number(D, rho_air, mu_air):
    """Calculate the Schmidt number."""

    if D <= 0.0 or rho_air <= 0.0 or mu_air <= 0.0:
        raise ValueError("D, rho_air and mu_air must be greater than zero")

    return mu_air / (rho_air * D)


# Stokes number

def stokes_number(dp_um, dd_um, mu_air):
    """Calculate the Stokes number for a dust particle and a raindrop."""

    if dd_um <= 0.0:
        raise ValueError("dd_um must be greater than zero")

    dd_m = dd_um * 1.0e-6
    Vt_rain = hydrometeor_terminal_velocity(dd_um, "rain")
    vt_particle = particle_terminal_velocity(dp_um, mu_air)
    tau = particle_relaxation_time(dp_um, mu_air)

    St = 2.0 * tau * (Vt_rain - vt_particle) / dd_m
    return St


# Critical Stokes number

def critical_stokes_number(Re):
    """Calculate the critical Stokes number S*."""

    if Re < 0.0:
        raise ValueError("Re cannot be negative")

    return (1.2 + np.log(1.0 + Re) / 12.0) / (1.0 + np.log(1.0 + Re))


# Brownian diffusion

def brownian_efficiency(Re, Sc):
    """Calculate the Brownian diffusion contribution to collection efficiency."""

    if Re <= 0.0 or Sc <= 0.0:
        raise ValueError("Re and Sc must be greater than zero")

    EB = (4.0 / (Re * Sc)) * (
        1.0
        + 0.4 * np.sqrt(Re) * Sc**(1.0 / 3.0)
        + 0.16 * np.sqrt(Re) * np.sqrt(Sc)
    )
    return EB


#  interception

def interception_efficiency(dp_um, dd_um, Re, mu_air, mu_water):
    """Calculate the interception contribution to collection efficiency."""

    if dd_um <= 0.0 or Re < 0.0 or mu_air <= 0.0 or mu_water <= 0.0:
        raise ValueError("diameters and viscosities must be positive")

    phi = dp_um / dd_um
    EI = 4.0 * phi * (
        mu_air / mu_water + phi * (1.0 + 2.0 * np.sqrt(Re))
    )
    return EI


# inertial impaction

def inertial_efficiency(dp_um, St, Sstar):
    """Calculate inertial impaction where dp > 2 um and St > S*."""

    if dp_um > 2.0 and St > Sstar:
        EII = (St - Sstar) / (St - Sstar + 2.0 / 3.0)
    else:
        EII = 0.0

    return EII


# total below-cloud collection efficiency

def collection_efficiency(
    dp_um,
    dd_um,
    temperature,
    rho_air,
    mu_air,
    mu_water,
):
    """Calculate collection efficiency and return the intermediate values."""

    D = particle_diffusivity(dp_um, temperature, mu_air)
    Re = reynolds_number(dd_um, rho_air, mu_air)
    Sc = schmidt_number(D, rho_air, mu_air)
    St = stokes_number(dp_um, dd_um, mu_air)
    Sstar = critical_stokes_number(Re)

    EB = brownian_efficiency(Re, Sc)
    EI = interception_efficiency(dp_um, dd_um, Re, mu_air, mu_water)
    EII = inertial_efficiency(dp_um, St, Sstar)
    E = EB + EI + EII

    return {
        "D": D,
        "Re": Re,
        "Sc": Sc,
        "St": St,
        "Sstar": Sstar,
        "EB": EB,
        "EI": EI,
        "EII": EII,
        "E": E,
    }


# Calculate the parameters 

def calculate_wet_deposition_parameters(dp_um, dc_um, dd_um, temperature, rho_air, mu_air, mu_water,):
    """Return the diagnostics needed to inspect one dust size bin."""

    Vt_cloud = hydrometeor_terminal_velocity(dc_um, "cloud")
    Vt_rain = hydrometeor_terminal_velocity(dd_um, "rain")
    Cc = cunningham_correction(dp_um)
    D = particle_diffusivity(dp_um, temperature, mu_air)
    vt = particle_terminal_velocity(dp_um, mu_air)
    tau = particle_relaxation_time(dp_um, mu_air)

    collection = collection_efficiency(dp_um, dd_um, temperature, rho_air, mu_air, mu_water)

    return {
        "Cc": Cc,
        "D": D,
        "vt": vt,
        "tau": tau,
        "Vt_cloud": Vt_cloud,
        "Vt_rain": Vt_rain,
        "Re": collection["Re"],
        "Sc": collection["Sc"],
        "St": collection["St"],
        "Sstar": collection["Sstar"],
        "EB": collection["EB"],
        "EI": collection["EI"],
        "EII": collection["EII"],
        "E": collection["E"] }


# in-cloud scavenging

def in_cloud_scavenging(precipitation, dc_um):
    """Calculate the in-cloud scavenging coefficient [s-1]."""

    if precipitation < 0.0 or dc_um <= 0.0:
        raise ValueError("precipitation cannot be negative and dc_um must be positive")

    E = 0.9
    Lambda = 4.2e-7 * E * precipitation / dc_um
    return Lambda


# Below-cloud scavenging : dd and the size-dependent E(dp)

def below_cloud_scavenging(dp_um, dd_um, precipitation, collection):
    """Calculate the below-cloud scavenging coefficient [s-1]."""

    if precipitation < 0.0 or dd_um <= 0.0:
        raise ValueError("precipitation cannot be negative and dd_um must be positive")

    E = collection["E"]
    Lambda = 4.2e-7 * E * precipitation / dd_um
    return Lambda

def update_dust_concentration(C_old, Lambda, dt):
    """Use the analytical solution C_new = C_old * exp(-Lambda * dt)."""

    if C_old < 0.0 or Lambda < 0.0 or dt < 0.0:
        raise ValueError("C_old, Lambda and dt cannot be negative")

    C_new = C_old * np.exp(-Lambda * dt)
    removed = C_old - C_new
    return C_new, removed


# Main wet-deposition calculation

def wet_deposition( dp_um, dc_um, dd_um, qcloud, P_grid, P_convective, temperature, rho_air, mu_air, mu_water, C_old, dt,):
    """Calculate wet removal for one grid cell, layer and dust size bin.

    P_grid and P_convective must be precipitation rates for this layer, in
    mm h-1. The paper identifies a layer as cloudy when QCLOUD > 0. If there
    is no precipitation, no wet removal is applied.
    """

    if qcloud < 0.0:
        raise ValueError("qcloud cannot be negative")
    if P_grid < 0.0 or P_convective < 0.0:
        raise ValueError("precipitation rates cannot be negative")

    P_total = P_grid + P_convective

    # Keep the intermediate values so they can be printed for checking.
    parameters = calculate_wet_deposition_parameters(
        dp_um,
        dc_um,
        dd_um,
        temperature,
        rho_air,
        mu_air,
        mu_water,
    )

    # The paper uses QCLOUD > 0 to identify a cloudy model layer.
    # Scavenging only occurs when precipitation is present.
    if P_total <= 0.0:
        Lambda = 0.0
        regime = "no wet deposition"
        E_actual = 0.0

    elif qcloud > 0.0:
        Lambda = in_cloud_scavenging(P_total, dc_um)
        regime = "in-cloud"
        E_actual = 0.9

    else:
        Lambda = below_cloud_scavenging(
            dp_um,
            dd_um,
            P_total,
            parameters,
        )
        regime = "below-cloud"
        E_actual = parameters["E"]

    C_new, removed = update_dust_concentration(C_old, Lambda, dt)

    parameters["qcloud"] = qcloud
    parameters["P_grid"] = P_grid
    parameters["P_convective"] = P_convective
    parameters["P_total"] = P_total
    parameters["regime"] = regime
    parameters["E_actual"] = E_actual
    parameters["Lambda"] = Lambda
    parameters["C_old"] = C_old
    parameters["C_new"] = C_new
    parameters["removed"] = removed

    return parameters




In [6]:
# Test calculation

if __name__ == "__main__":

    temperature = 280.0      # K
    rho_air = 1.20           # kg m-3
    mu_air = 1.75e-5          # Pa s
    mu_water = 1.0e-3         # Pa s

    # Example layer-specific precipitation rates (mm h-1).
    # Replace these values with the corresponding model output.
    P_grid = 10.0
    P_convective = 04.0

    dc = 20.0                 # mean cloud droplet diameter [um]
    dd = 200.0                # mean raindrop diameter [um]

    qcloud = 1.0e-4           # QCLOUD [kg kg-1], from Morrison microphysics
    C_initial = 1.0            # example initial dust concentration
    dt = 150.0                 # WRF time step in the paper [s]

    print("WET DEPOSITION")
    print("=" * 80)

    for dp in DUST_DIAMETERS_UM:
        result = wet_deposition(
            dp_um=dp,
            dc_um=dc,
            dd_um=dd,
            qcloud=qcloud,
            P_grid=P_grid,
            P_convective=P_convective,
            temperature=temperature,
            rho_air=rho_air,
            mu_air=mu_air,
            mu_water=mu_water,
            C_old=C_initial,
            dt=dt,
        )

        print(
            f"dp = {dp:5.2f} um   "
            f"QCLOUD = {qcloud:.3e}   "
            f"regime = {result['regime']:18s}   "
            f"Lambda = {result['Lambda']:.4e} s^-1   "
            f"C_new = {result['C_new']:.6f}   "
            f"removed = {result['removed']:.6f}"
        )

    # Check the four cloud/precipitation combinations for one dust size.
    print("\nCHECKING THE FOUR CONDITIONS")
    print("=" * 80)

    test_cases = [
        ("cloud + precipitation", 1.0e-4, 10.0, 4.0),
        ("cloud, no precipitation", 1.0e-4, 0.0, 0.0),
        ("no cloud + precipitation", 0.0, 10.0, 4.0),
        ("no cloud, no precipitation", 0.0, 0.0, 0.0),
    ]

    for case_name, case_qcloud, case_P_grid, case_P_convective in test_cases:
        result = wet_deposition(
            dp_um=4.8,
            dc_um=dc,
            dd_um=dd,
            qcloud=case_qcloud,
            P_grid=case_P_grid,
            P_convective=case_P_convective,
            temperature=temperature,
            rho_air=rho_air,
            mu_air=mu_air,
            mu_water=mu_water,
            C_old=C_initial,
            dt=dt,
        )
        print(f"{case_name:28s} -> {result['regime']}")

    # Print the intermediate values for dp = 4.8 um.
    dp = 4.8
    result = wet_deposition(
        dp_um=dp,
        dc_um=dc,
        dd_um=dd,
        qcloud=qcloud,
        P_grid=P_grid,
        P_convective=P_convective,
        temperature=temperature,
        rho_air=rho_air,
        mu_air=mu_air,
        mu_water=mu_water,
        C_old=C_initial,
        dt=dt,
    )

    print("\nCALCULATED PARAMETERS FOR dp = 4.8 um")
    print("=" * 80)
    print(f"Cc       = {result['Cc']:.6e}")
    print(f"D        = {result['D']:.6e}")
    print(f"vt       = {result['vt']:.6e}")
    print(f"tau      = {result['tau']:.6e}")
    print(f"Vt cloud = {result['Vt_cloud']:.6e}")
    print(f"Vt rain  = {result['Vt_rain']:.6e}")
    print(f"Re       = {result['Re']:.6e}")
    print(f"Sc       = {result['Sc']:.6e}")
    print(f"St       = {result['St']:.6e}")
    print(f"Sstar    = {result['Sstar']:.6e}")
    print(f"EB       = {result['EB']:.6e}")
    print(f"EI       = {result['EI']:.6e}")
    print(f"EII      = {result['EII']:.6e}")
    print(f"E        = {result['E']:.6e}")
    print(f"P total  = {result['P_total']:.6e} mm h-1")
    print(f"QCLOUD   = {result['qcloud']:.6e} kg kg-1")
    print(f"Regime   = {result['regime']}")
    print(f"Lambda   = {result['Lambda']:.6e} s-1")
    print(f"C new    = {result['C_new']:.6e}")
    print(f"Removed  = {result['removed']:.6e}")


WET DEPOSITION
dp =  1.46 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 2.6460e-07 s^-1   C_new = 0.999960   removed = 0.000040
dp =  2.80 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 2.6460e-07 s^-1   C_new = 0.999960   removed = 0.000040
dp =  4.80 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 2.6460e-07 s^-1   C_new = 0.999960   removed = 0.000040
dp =  9.00 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 2.6460e-07 s^-1   C_new = 0.999960   removed = 0.000040
dp = 16.00 um   QCLOUD = 1.000e-04   regime = in-cloud             Lambda = 2.6460e-07 s^-1   C_new = 0.999960   removed = 0.000040

CHECKING THE FOUR CONDITIONS
cloud + precipitation        -> in-cloud
cloud, no precipitation      -> no wet deposition
no cloud + precipitation     -> below-cloud
no cloud, no precipitation   -> no wet deposition

CALCULATED PARAMETERS FOR dp = 4.8 um
Cc       = 1.034792e+00
D        = 5.052977e-12
vt       = 1.282246e+10